# BMO robust coke-rate model: test notebook

Tests the Data-Driven coke-rate model on `furnace_dataset.csv`, through the same code the BMO page runs (`utils.bmo.direct_coke_model` and `utils.bmo.robust_coke_target`). Nothing here changes the live deployment: retraining writes to a temporary folder.

| Section | What it shows |
|---|---|
| 1 | The output being predicted, built from your dataset |
| 2 | Why hourly `COKE_CALC_MT / production` cannot be the target |
| 3 | Data quality: undercount spells, the one-sided noise in `COKE_CALC_MT`, the set-point offset, the noise floor |
| 4 | The deployed model against the target |
| 5 | Holdout validation with the app's own gates |
| 6 | Rolling backtest: XGBoost vs Ridge, OLS, GP regression, persistence, ARIMA, ARIMAX |
| 7 | What-if: PCI and nut coke response at today's state |
| 8 | Your own PCI / nut coke values |

**Kernel:** the project `.venv` (it needs `ipykernel`: `uv pip install ipykernel`). **Runtime:** about 5 minutes, mostly section 6; set `RUN_BACKTEST = False` to skip it.

In [ ]:
from pathlib import Path
import sys, tempfile, time, warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
try:
    display
except NameError:  # running as a plain script
    display = print

# Walk up to the repository root so the notebook runs from any folder in the repo.
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "utils" / "bmo").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the evonith_webapp repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from utils.bmo import direct_coke_model as dcm
from utils.bmo.robust_coke_target import (
    NUT_COKE_FEATURE, PCI_FEATURE, TARGET_COLUMN, RobustCokeSettings,
    fe_charged_mt, robust_coke_target, training_columns, trusted_coke_hours,
)

DATASET = ROOT / "src/assets/data/furnace_dataset.csv"  # <- the furnace_dataset input
BUNDLE = ROOT / "src/assets/models/bmo_coke_robust"      # shipped model + target config
DEPLOYMENTS = ROOT / "src/storage/bmo_coke_model"        # app retrains (active.json)
RUN_BACKTEST = True                                      # section 6, ~3-4 minutes

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3})
print(f"dataset: {DATASET} ({DATASET.stat().st_size / 1e6:.1f} MB)")

## 1. The output being predicted

For each hour $t$, over the trailing 24 hours $W$ of **trusted** hours (section 3):

$$
\text{coke rate}(t) = 1000 \times \underbrace{\frac{\sum_{W} \text{COKE\_CALC\_MT}}{\sum_{W} \text{Fe charged}}}_{\text{coke-to-iron ratio of the charges made}} \times \underbrace{\frac{\sum_{t \pm 84\,h} \text{Fe charged}}{\sum_{t \pm 84\,h} \text{PRODUCTIONTONNESPERHR}}}_{\text{Fe per tonne HM, one week}} \quad [\text{kg/THM}]
$$

Fe charged = ore x (1 - TM%) x Fe(T)% + sinter x Fe(T)% + pellet x (1 - TM%) x Fe2O3% x 0.6994. Coke and ore are counted on the same charges, so the charge count cancels in the first factor. The second factor puts the rate back on the plant's production basis. The features describe the same 24-hour window.

In [ ]:
config = dcm._read_json(BUNDLE / "config.json")
settings = RobustCokeSettings.from_config(config)
t0 = time.time()
# Same cleaning and window features as the app.
cleaned, audit, X, cfg, _ = dcm._prepare_context(DATASET, config)
target = robust_coke_target(cleaned, audit, settings)
y = target[TARGET_COLUMN]
ok = audit["normal_eligible"] & audit["in_requested_window"] & y.notna()
print(f"prepared {len(cleaned):,} hours in {time.time() - t0:.0f}s "
      f"({cleaned.index.min():%d %b %Y} to {cleaned.index.max():%d %b %Y %H:%M} UTC); "
      f"{ok.sum():,} hours have a trusted {settings.window_hours}-h target")
display(target.loc[ok, [TARGET_COLUMN, "coke_per_fe_t", "fe_per_thm_t"]].describe().round(3))

## 2. Why not hourly `COKE_CALC_MT / production`

Hourly production runs on a different clock from charging: it barely tracks the Fe charged in the same hour, yet agrees with it over a day. Coke and ore share one charge count, so the hourly coke rate mostly tracks the same-hour burden rate. A model fed per-THM burden features then scores a high R2 by predicting the charge count. The reported `COKE RATE KG/THM` is a set point that rarely changes.

In [ ]:
coke = cleaned["COKE_CALC_MT"]
prod = cleaned["PRODUCTIONTONNESPERHR"]
fe = fe_charged_mt(cleaned)
run = audit["normal_eligible"] & fe.gt(0) & coke.gt(0)
hourly = (1000 * coke / prod).where(run)
burden = cleaned[["ORE_CALC_MT", "SINTER_CALC_MT", "TOTAL_PELLET_CALC_MT"]].fillna(0).sum(axis=1)
burden_thm = (1000 * burden / prod).where(run)
setpoint = cleaned["COKE RATE KG/THM"].where(run)
trusted = trusted_coke_hours(cleaned, audit, settings)  # section 3 explains the screen


def day_sum(series):
    return series.where(run).rolling(24, min_periods=18).sum()


summary = pd.Series({
    "hour-to-hour |change|, hourly COKE_CALC/production (kg/THM)": hourly.diff().abs().median(),
    "hour-to-hour |change|, robust 24-h target (kg/THM)": y.where(ok).diff().abs().median(),
    "share of hours the set point changes": (setpoint.diff().abs() > 0.5).mean(),
    "corr(hourly coke rate, same-hour burden kg/THM), trusted hours":
        hourly.where(trusted).corr(burden_thm.where(trusted)),
    "corr of their hour-to-hour changes, trusted hours":
        hourly.where(trusted).diff().corr(burden_thm.where(trusted).diff()),
    "corr(production, Fe charged), hourly": prod.where(run).corr(fe.where(run)),
    "corr(production, Fe charged), 24-h sums": day_sum(prod).corr(day_sum(fe)),
})
display(summary.round(3).to_frame("value"))

start = cleaned.index.max() - pd.Timedelta(days=30)
fig, ax = plt.subplots()
ax.plot(hourly.loc[start:], lw=0.6, alpha=0.5, label="hourly 1000 x COKE_CALC_MT / production")
ax.plot(setpoint.loc[start:], lw=1.5, label="COKE RATE KG/THM (set point)")
ax.plot(y.where(ok).loc[start:], lw=2.2, label="robust 24-h target")
ax.set_ylim(150, 500)
ax.set_ylabel("kg/THM")
ax.set_title("Last 30 days")
ax.legend(loc="upper left")
plt.show()

## 3. Data quality and the noise in `COKE_CALC_MT`

Three things are checked:

1. **Multi-day undercounts.** Days whose coke-mass rate disagrees with the set point by more than -20% / +25% are dropped (1 Nov - 5 Dec 2025 reads about half the real coke). The set point is used only for this check.
2. **Single-hour weighing spikes.** Hours whose coke/Fe is more than 6 robust sigma from its week are dropped.
3. **One-sided error.** Hourly coke/Fe deviations are skewed upward, and the mass-based rate sits above the set point on almost every day. The cells below test whether the upward hours are counting noise or real coke, and what a noise-robust estimator would change.

In [ ]:
agreement =(1000 * day_sum(coke) / day_sum(prod)) / setpoint.rolling(24, min_periods=18).mean()
fig, ax = plt.subplots()
ax.plot(agreement, lw=1, label="24-h coke-mass rate / set point")
ax.axhspan(settings.setpoint_ratio_min, settings.setpoint_ratio_max, color="green", alpha=0.08, label="kept band")
ax.set_ylim(0.3, 1.5)
ax.set_title("Undercount spells are dropped from the target")
ax.legend()
plt.show()
print(f"running hours {run.sum():,}; trusted {trusted.sum():,}; dropped {(run & ~trusted).sum():,}")

In [ ]:
ratio = (coke / fe).where(trusted)
local = lambda s, hours: s.rolling(hours, min_periods=12, center=True).median()
rel_week = (ratio / local(ratio, 168) - 1).dropna()
rel_day = (ratio / local(ratio, 25) - 1).dropna()
display(pd.DataFrame({
    "vs 7-day median": [(rel_week > .10).mean(), (rel_week < -.10).mean(), rel_week.skew()],
    "vs 24-h median": [(rel_day > .10).mean(), (rel_day < -.10).mean(), rel_day.skew()],
}, index=["share of hours > +10%", "share of hours < -10%", "skew"]).round(3))

offset = (y - setpoint.rolling(24, min_periods=18).mean()).where(ok).iloc[::24].dropna()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(rel_day.clip(-0.2, 0.3), bins=100)
ax[0].set_title("Hourly coke/Fe vs its 24-h median")
ax[1].hist(offset, bins=40)
ax[1].axvline(0, color="k")
ax[1].set_title(f"Robust target - set point: above on {(offset > 0).mean():.0%} of days, "
                f"median {offset.median():+.1f} kg/THM")
plt.show()

# Noise or real coke? Look at what changed per charge in the upward hours.
charges = cleaned["CHARGES/HRS."].where(trusted)
coke_per_charge, fe_per_charge = coke / charges, fe / charges
high = rel_day.index[rel_day > 0.08]
print(f"in hours > +8%: coke per charge {(coke_per_charge / local(coke_per_charge, 25) - 1)[high].median():+.1%}, "
      f"Fe per charge {(fe_per_charge / local(fe_per_charge, 25) - 1)[high].median():+.1%}; "
      f"the next hour is still {rel_day.shift(-1)[rel_day > 0.08].mean():+.1%} "
      "(a charge-boundary artefact would reverse)")

sum_ratio = coke.where(trusted).rolling(24, min_periods=18).sum() / fe.where(trusted).rolling(24, min_periods=18).sum()
median_ratio = ratio.rolling(24, min_periods=18).median()
print(f"median of hourly ratios instead of the sum moves the 24-h level by {(median_ratio / sum_ratio - 1).median():+.2%}")
noise = (y / y.rolling(72, center=True, min_periods=48).mean() - 1).where(ok).std()
print(f"day-to-day noise left in the target: {noise:.2%} = {noise * y.median():.1f} kg/THM "
      "(a floor under any model's later-date MAE)")

## 4. The deployed model

Loads whatever the BMO page would use: the newest deployment in `src/storage/bmo_coke_model` if one is active, otherwise the shipped bundle. The chart covers hours the active model was trained on, so it shows fit, not validation. Section 5 gives the holdout scores.

In [ ]:
svc = dcm.DirectCokeModelService(bundled_dir=BUNDLE, deployment_dir=DEPLOYMENTS,
                                 max_stale_hours=1e9, max_source_age_hours=1e9)
status = svc.status()
print(f"active model: {status['deployment_id']} | task {status['task']} | "
      f"{status['feature_count']} features | {status['training_rows']:,} training windows")
display(pd.DataFrame({k: pd.Series(v) for k, v in status["validation"].items()
                      if isinstance(v, dict)}).round(3))

history = svc.prediction_history(DATASET, history_days=14)
fig, ax = plt.subplots()
ax.plot(history["raw_predicted_coke_kg_per_thm"], label="prediction")
ax.plot(history["actual_coke_kg_per_thm"], ".", ms=3, label="robust target")
ax.set_ylabel("kg/THM")
ax.set_title("Last 14 days (in-sample)")
ax.legend()
plt.show()

## 5. Holdout validation with the app's gates

Runs the exact retrain the Model Accuracy button and the daily auto-retrain run, into a temporary folder, so nothing is deployed.

- **Random-day holdout:** 20% of whole days, purged by one window, so overlapping windows cannot leak.
- **Later-time holdout:** the last 14 days, never in the fit. The gates are MAE and skill over carrying last week's level forward. R2 is reported but not gated, because a quiet fortnight has little variance to explain.
- **PCI response:** coke change for PCI -40 kg/THM on the later-time rows.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    report = dcm.retrain_and_maybe_deploy(DATASET, bundled_dir=BUNDLE, deployment_dir=Path(tmp))
print("passes every gate:", report.passed, "|", "; ".join(report.reasons) or "no failures")
display(pd.DataFrame({"random-day holdout": pd.Series(report.random_metrics),
                      "later-time holdout": pd.Series(report.later_time_metrics)}).round(3))
display(pd.Series(report.fuel_response).round(3).to_frame("fuel response"))

## 6. Rolling backtest: is another model better on later dates?

Each model is refit weekly on everything before that week (24-hour purge) and scored on the following week, across the last 18 weeks. There are two kinds of model:

- **Conditional models** (XGBoost, Ridge, OLS, GP regression) see only that window's conditions. They are the only kind that can answer *"what if PCI is 160?"*
- **History-based forecasters** (persistence, ARIMA, ARIMAX) predict the next plant day from the coke rates already measured. They score well because yesterday's rate is a strong guide to today's, but they cannot estimate a what-if. ARIMAX's PCI coefficient is an independent check on the PCI response.

Daily points are the windows ending 06:00 IST (00:30 UTC), one per plant day.

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

Xo, yo = X.loc[ok], y.loc[ok].astype(float)
PHYSICAL = [PCI_FEATURE, NUT_COKE_FEATURE, "slag_kg_thm_repo__win", "SINTER_SHARE__win",
            "TOTAL_PELLET_SHARE__win", "BURDEN_PER_FE__win", "HOT BLAST TEMP.OC__win",
            "O2 ENRICHMENT %__win", "PRODUCTIONTONNESPERHR__win", "HOT BLAST VOLUMENM3/HR.__win",
            "STEAMKGS/HR.__win", "COKE_ASH%__assay"]


def physical(frame):
    return [c for c in PHYSICAL if c in frame and frame[c].notna().mean() > 0.5]


def fit_xgb(Xtr, ytr):  # the deployed recipe: all window features, monotone PCI / nut coke
    cols = training_columns(Xtr)
    model = dcm._train_model(Xtr, ytr, cols, depth=3, rounds=300, seed=20260922)
    return lambda Xte: dcm._predict_model(model, Xte, cols)


def fit_ridge(Xtr, ytr):
    cols = training_columns(Xtr)
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-2, 4, 25), cv=TimeSeriesSplit(5))).fit(Xtr[cols], ytr)
    return lambda Xte: pipe.predict(Xte[cols])


def fit_ols(Xtr, ytr):
    cols = physical(Xtr)
    pipe = make_pipeline(SimpleImputer(strategy="median"), LinearRegression()).fit(Xtr[cols], ytr)
    return lambda Xte: pipe.predict(Xte[cols])


def fit_gp(Xtr, ytr, step=6):  # every 6th hour: neighbouring windows are near-duplicates
    cols = physical(Xtr)
    Xs, ys = Xtr[cols].iloc[::step], ytr.iloc[::step]
    pre = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit(Xs)
    kernel = (ConstantKernel(1.0) * RBF(np.full(len(cols), 3.0), (0.1, 100))
              + WhiteKernel(0.1, (1e-3, 10)))
    gp = GaussianProcessRegressor(kernel, normalize_y=True, random_state=0).fit(pre.transform(Xs), ys)
    return lambda Xte: gp.predict(pre.transform(Xte[cols]))


CONDITIONAL = {"XGBoost (deployed)": fit_xgb, "Ridge (all features)": fit_ridge,
               "OLS (12 physical features)": fit_ols, "GP regression (12 features)": fit_gp}


def plant_days(series):
    return series[series.index.hour == 0].asfreq("D")

In [ ]:
if RUN_BACKTEST:
    t0 = time.time()
    purge = pd.Timedelta(hours=24)
    end = yo.index.max().floor("D")
    weeks = pd.date_range(end - pd.Timedelta(days=126), end, freq="7D", inclusive="left")
    preds = {name: pd.Series(np.nan, index=yo.index) for name in CONDITIONAL}
    pci_resp = {name: [] for name in CONDITIONAL}
    for week in weeks:
        train = yo.index < week - purge
        test = (yo.index >= week) & (yo.index < week + pd.Timedelta(days=7))
        if not test.any():
            continue
        for name, fit in CONDITIONAL.items():
            predict = fit(Xo[train], yo[train])
            preds[name][test] = predict(Xo[test])
            shifted = Xo[test].copy()
            shifted[PCI_FEATURE] -= 40
            pci_resp[name].append(float(np.mean(predict(shifted) - predict(Xo[test]))))

    yd = plant_days(yo)
    scored = (yd.index >= weeks[0]) & yd.notna()
    forecasts = {"Persistence (yesterday)": yd.shift(1),
                 "ARIMA(1,1,1)": pd.Series(np.nan, index=yd.index),
                 "ARIMAX(1,1,1) + PCI/nut/slag/sinter": pd.Series(np.nan, index=yd.index)}
    exog = Xo.reindex(yd.index)[[PCI_FEATURE, NUT_COKE_FEATURE, "slag_kg_thm_repo__win",
                                 "SINTER_SHARE__win"]].interpolate(limit_direction="both")
    arimax_pci = []
    for week in weeks:
        history_days = yd[yd.index < week]
        in_week = (yd.index >= week) & (yd.index < week + pd.Timedelta(days=7))
        arima = SARIMAX(history_days, order=(1, 1, 1)).fit(disp=False).apply(yd, refit=False)
        forecasts["ARIMA(1,1,1)"][in_week] = arima.get_prediction().predicted_mean[in_week]
        arimax = SARIMAX(history_days, exog=exog.loc[history_days.index], order=(1, 1, 1)).fit(disp=False)
        arimax_pci.append(arimax.params[PCI_FEATURE])
        rolled = arimax.apply(yd, exog=exog, refit=False)
        forecasts["ARIMAX(1,1,1) + PCI/nut/slag/sinter"][in_week] = rolled.get_prediction().predicted_mean[in_week]

    rows, weekly = [], {}
    for name, pred in preds.items():
        err = (plant_days(pred) - yd)[scored].dropna()
        hourly_err = (pred - yo)[yo.index >= weeks[0]].abs()
        weekly[name] = hourly_err.groupby(hourly_err.index.floor("7D")).mean()
        rows.append({"model": name, "kind": "conditional (can do what-ifs)", "daily MAE": err.abs().mean(),
                     "bias": err.mean(), "worst week MAE": weekly[name].max(),
                     "coke for PCI -40": np.mean(pci_resp[name]), "days": len(err)})
    for name, pred in forecasts.items():
        err = (pred - yd)[scored].dropna()
        rows.append({"model": name, "kind": "uses yesterday's measured coke", "daily MAE": err.abs().mean(),
                     "bias": err.mean(), "worst week MAE": np.nan,
                     "coke for PCI -40": -40 * np.mean(arimax_pci) if "ARIMAX" in name else np.nan,
                     "days": len(err)})
    board = pd.DataFrame(rows).set_index("model")
    print(f"{len(weeks)} weekly refits, {weeks[0]:%d %b} to {end:%d %b %Y}; target sd {yo[yo.index >= weeks[0]].std():.1f} kg/THM; "
          f"took {time.time() - t0:.0f}s")
    display(board.round(2))

    fig, ax = plt.subplots()
    for name, series in weekly.items():
        ax.plot(series.index, series.values, "o-", label=name)
    ax.set_yscale("log")
    ax.set_ylabel("weekly MAE, kg/THM (log)")
    ax.set_title("Linear models fail badly in weeks whose conditions leave the training range")
    ax.legend()
    plt.show()

## 7. What-if: PCI and nut coke at today's state

Runs the same call the BMO page makes: the operator's PCI and nut coke rates are written over every hour the latest 24-hour windows use, the window features are rebuilt, and the prediction is the median of the last 6 hourly windows. A value outside the training p01-p99 range is flagged; above about 205 kg/THM PCI the response flattens because the plant has not run there.

In [ ]:
def what_if(pci, nut):
    return svc.predict_from_history(DATASET, pci_kg_per_thm=pci, nut_coke_kg_per_thm=nut, lookback_hours=6)


pci_curve = pd.DataFrame([
    {"PCI kg/THM": pci, "coke kg/THM": (p := what_if(pci, 75.0)).value_kg_per_thm,
     "outside training range": ", ".join(p.outside_training_p01_p99)}
    for pci in range(140, 221, 10)
])
nut_curve = pd.DataFrame([
    {"nut coke kg/THM": nut, "coke kg/THM": what_if(180.0, nut).value_kg_per_thm}
    for nut in range(55, 91, 5)
])
display(pci_curve.round(1))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(pci_curve["PCI kg/THM"], pci_curve["coke kg/THM"], "o-")
ax[0].set_xlabel("PCI kg/THM (nut coke 75)")
ax[0].set_ylabel("predicted coke kg/THM")
ax[1].plot(nut_curve["nut coke kg/THM"], nut_curve["coke kg/THM"], "o-")
ax[1].set_xlabel("nut coke kg/THM (PCI 180)")
plt.show()
by_pci = pci_curve.set_index("PCI kg/THM")["coke kg/THM"]
print(f"PCI 200 -> 160: coke {by_pci[200]:.1f} -> {by_pci[160]:.1f} kg/THM ({by_pci[160] - by_pci[200]:+.1f}), "
      f"{(by_pci[160] - by_pci[200]) / 40:.2f} kg coke per kg PCI")

## 8. Your own values

In [ ]:
MY_PCI, MY_NUT_COKE = 170.0, 70.0  # kg/THM
p = what_if(MY_PCI, MY_NUT_COKE)
print(f"PCI {MY_PCI:.0f}, nut coke {MY_NUT_COKE:.0f} kg/THM -> coke {p.value_kg_per_thm:.1f} kg/THM "
      f"(windows ending {p.window_end_utc} UTC; usable={p.usable}; "
      f"{'; '.join(p.reasons) or 'no freshness warnings'}; "
      f"outside training range: {', '.join(p.outside_training_p01_p99) or 'none'})")